# Analisis Sentimen Deteksi Judi Online (Judol)
## Perbandingan XGBoost vs Algoritma ML Klasik
**Dataset:** 1.508 data berlabel (3 anotator, majority vote)  
**Kelas:** Positif | Netral | Negatif  
**Fitur:** TF-IDF Unigram / Bigram / Trigram


## 1. Instalasi & Import Library

In [ ]:
!pip install xgboost wordcloud -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)
from sklearn.base import clone
from xgboost import XGBClassifier

MODEL_PALETTE = ['#2196F3','#9C27B0','#FF9800','#4CAF50','#F44336','#FF6F00']
print(" Library berhasil diimport")


## 2. Load & Gabung Data Anotator (Majority Vote)

In [ ]:
import pandas as pd
from collections import Counter

a1 = pd.read_csv('/content/anotator_1.csv')
a2 = pd.read_csv('/content/anotator_2.csv')
a3 = pd.read_csv('/content/anotator_3.csv')

def majority_vote(row):
    return Counter(row).most_common(1)[0][0]

labels_df = pd.DataFrame({'a1': a1['label_anotator'],
                           'a2': a2['label_anotator'],
                           'a3': a3['label_anotator']})
majority = labels_df.apply(majority_vote, axis=1)

df = pd.DataFrame({'text': a1['TEXT_SIAP'], 'label': majority}).dropna().reset_index(drop=True)
df['text']     = df['text'].astype(str)
df['text_len'] = df['text'].str.split().str.len()

print(f"Total data : {df.shape[0]} baris")
print("\nDistribusi label (majority vote):")
print(df['label'].value_counts())
df.head(3)

## 2.5 Analisis Kualitas Dataset

In [ ]:
counts = df['label'].value_counts()
total  = len(df)
pct    = (counts / total * 100).round(2)
imbalance_ratio = counts.max() / counts.min()

print("=== DISTRIBUSI & PERSENTASE KELAS ===")
dist_df = pd.DataFrame({'Jumlah': counts, 'Persentase (%)': pct})
display(dist_df)
print(f"\nImbalance Ratio (mayoritas/minoritas): {imbalance_ratio:.2f}:1")

print("\n=== STATISTIK PANJANG DOKUMEN (token) ===")
stats = df['text_len'].describe().round(2)
print(f"  Minimum   : {int(stats['min'])} token")
print(f"  Maksimum  : {int(stats['max'])} token")
print(f"  Rata-rata : {stats['mean']:.1f} token")
print(f"  Median    : {stats['50%']:.0f} token")
print(f"  Std Dev   : {stats['std']:.1f}")

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in zip(['negatif','netral','positif'], ['#F44336','#2196F3','#4CAF50']):
    ax.hist(df[df['label']==label]['text_len'], bins=30, alpha=0.6,
            label=label.capitalize(), color=color, edgecolor='white')
ax.axvline(df['text_len'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f"Mean={df['text_len'].mean():.1f}")
ax.set_title('Distribusi Panjang Dokumen per Kelas Sentimen', fontweight='bold')
ax.set_xlabel('Jumlah Token'); ax.set_ylabel('Frekuensi'); ax.legend()
plt.tight_layout(); plt.show()

all_words = ' '.join(df['text']).split()
vocab = set(all_words)
print(f"\n=== KUALITAS VOCABULARY ===")
print(f"Total token       : {len(all_words):,}")
print(f"Token unik (vocab): {len(vocab):,}")
print(f"Type-Token Ratio  : {len(vocab)/len(all_words):.4f}")

print("\n=== OUTLIER ===")
short = df[df['text_len'] <= 2]
long_ = df[df['text_len'] >= 50]
print(f"Dokumen sangat pendek (<=2 token): {len(short)}")
print(f"Dokumen sangat panjang (>=50 token): {len(long_)}")
if len(short) > 0:
    display(short[['text','label','text_len']].head(5))


## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribusi Kelas & Panjang Teks

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('EDA - Dataset Judol (Majority Vote Anotator)', fontsize=13, fontweight='bold')

counts = df['label'].value_counts()
colors = ['#F44336','#2196F3','#4CAF50']

bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+5, str(val), ha='center', fontweight='bold')
axes[0].set_title('Distribusi Kelas', fontweight='bold'); axes[0].set_ylabel('Jumlah')

axes[1].pie(counts.values, labels=counts.index, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Proporsi Kelas', fontweight='bold')

for label, color in zip(['negatif','netral','positif'], colors):
    axes[2].hist(df[df['label']==label]['text_len'], bins=30,
                 alpha=0.6, label=label, color=color)
axes[2].set_title('Panjang Token per Kelas', fontweight='bold')
axes[2].set_xlabel('Jumlah Token'); axes[2].legend()

plt.tight_layout(); plt.show()
print("\nRata-rata panjang token per kelas:")
print(df.groupby('label')['text_len'].mean().round(1))


### 3.2 WordCloud per Kelas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('WordCloud per Kelas Sentimen', fontsize=13, fontweight='bold')

for ax, (label, cmap) in zip(axes, [('positif','Greens'),('netral','Blues'),('negatif','Reds')]):
    text = ' '.join(df[df['label']==label]['text'])
    wc = WordCloud(width=500, height=280, background_color='white', colormap=cmap,
                   max_words=80, collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title(f'{label.upper()}', fontweight='bold', fontsize=13)

plt.tight_layout(); plt.show()


### 3.3 Top 10 Kata per Kelas

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('Frekuensi Unigram & Bigram per Kelas', fontsize=13, fontweight='bold')

for row_idx, (label, color) in enumerate([('positif','#4CAF50'),('netral','#2196F3'),('negatif','#F44336')]):
    texts = df[df['label']==label]['text'].tolist()
    for col_idx, (ng, ng_label) in enumerate([(1,'Unigram'),(2,'Bigram')]):
        vec = CountVectorizer(max_features=10, ngram_range=(ng,ng))
        X_v = vec.fit_transform(texts)
        freq = pd.Series(X_v.toarray().sum(axis=0), index=vec.get_feature_names_out()).sort_values()
        axes[row_idx][col_idx].barh(freq.index, freq.values, color=color, edgecolor='white', alpha=0.85)
        axes[row_idx][col_idx].set_title(f'{label.upper()} - Top 10 {ng_label}', fontweight='bold', fontsize=10)

plt.tight_layout(); plt.show()


### 3.4 Tabel Kuantitatif Preprocessing

In [ ]:

try:
    df_clean = pd.read_csv('/content/cleaned_data judol.csv')
    df_norm  = pd.read_csv('/content/normalisasi_datajudol.csv')
    df_stop  = pd.read_csv('/content/stopword_datajudol.csv')
    df_stem  = pd.read_csv('/content/stemming_datajudol.csv')

    def avg_tok(s): return s.dropna().astype(str).apply(lambda x: len(x.split())).mean()
    def tot_tok(s): return s.dropna().astype(str).apply(lambda x: len(x.split())).sum()

    raw_avg = avg_tok(df_clean['full_text'])
    raw_tot = tot_tok(df_clean['full_text'])

    rows = [
        ('Raw Text (sebelum cleaning)',   raw_avg,                          raw_tot),
        ('Cleaning (hapus noise/URL)',     avg_tok(df_clean['full_text']),   tot_tok(df_clean['full_text'])),
        ('Normalisasi (kamus singkatan)',  avg_tok(df_norm['corrected_text']),tot_tok(df_norm['corrected_text'])),
        ('Stopword Removal',              avg_tok(df_stop['stop_word']),    tot_tok(df_stop['stop_word'])),
        ('Stemming (TEXT SIAP)',           avg_tok(df_stem['stemming']),     tot_tok(df_stem['stemming'])),
    ]
    tbl_pre = pd.DataFrame(rows, columns=['Tahapan','Rata-rata Token/Dok','Total Token'])
    tbl_pre['Reduksi (%)'] = ((1 - tbl_pre['Total Token'] / raw_tot) * 100).round(1)
    tbl_pre['Rata-rata Token/Dok'] = tbl_pre['Rata-rata Token/Dok'].round(1)

    print("=== DAMPAK TAHAPAN PREPROCESSING ===")
    display(tbl_pre.style
            .hide(axis='index')
            .format({'Rata-rata Token/Dok': '{:.1f}', 'Total Token': '{:,}', 'Reduksi (%)': '{:.1f}%'})
            .background_gradient(cmap='Blues_r', subset=['Total Token'])
            .set_caption('Perubahan Jumlah Token per Tahap Preprocessing'))

    print(f"\nTotal reduksi: {raw_tot:,} -> {int(tbl_pre['Total Token'].iloc[-1]):,} token")
    print(f"Efisiensi preprocessing: {tbl_pre['Reduksi (%)'].iloc[-1]:.1f}% token berkurang")

except FileNotFoundError as e:
    print(f"[INFO] File tidak ditemukan: {e}")
    print("Jalankan cell ini setelah meng-upload file CSV preprocessing ke Colab.")
    print(f"\n[Fallback] Rata-rata token TEXT_SIAP yang tersedia: {df['text_len'].mean():.1f}")
    print(f"Total token TEXT_SIAP: {df['text_len'].sum():,}")


### 3.5 Distribusi Sentimen - Jumlah & Persentase

In [ ]:
counts = df['label'].value_counts()
pct    = (counts / len(df) * 100).round(2)
colors = ['#F44336','#2196F3','#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribusi Sentimen - Jumlah & Persentase', fontsize=13, fontweight='bold')

bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val, p in zip(bars, counts.values, pct.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+4,
                 f'{val}\n({p}%)', ha='center', va='bottom',
                 fontweight='bold', fontsize=11)
axes[0].set_title('Jumlah Data per Kelas', fontweight='bold')
axes[0].set_ylabel('Jumlah Data')
axes[0].set_ylim(0, counts.max()*1.2)

wedges, _, autotexts = axes[1].pie(
    counts.values, labels=counts.index, colors=colors,
    autopct='%1.2f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2},
    textprops={'fontsize':11})
for at in autotexts:
    at.set_fontweight('bold')
axes[1].add_patch(plt.Circle((0,0), 0.60, fc='white'))
axes[1].set_title('Proporsi Kelas Sentimen', fontweight='bold')
axes[1].text(0, 0, f'n={len(df)}', ha='center', va='center', fontsize=13, fontweight='bold')

plt.tight_layout(); plt.show()

print("\n=== RINGKASAN DISTRIBUSI SENTIMEN ===")
summary = pd.DataFrame({
    'Kelas'         : counts.index,
    'Jumlah'        : counts.values,
    'Persentase (%)'  : pct.values
}).reset_index(drop=True)
display(summary.style.hide(axis='index').format({'Persentase (%)': '{:.2f}%'}))
print(f"Imbalance Ratio: {counts.max()}/{counts.min()} = {counts.max()/counts.min():.2f}:1")


### 3.6 Visualisasi Bobot TF-IDF Rata-Rata (Bukan Frekuensi Mentah)

In [ ]:
tfidf_viz = TfidfVectorizer(max_features=2000, ngram_range=(1,1), min_df=2)
X_viz      = tfidf_viz.fit_transform(df['text'])
feats_viz  = np.array(tfidf_viz.get_feature_names_out())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Bobot TF-IDF Rata-rata (mencerminkan kepentingan kata, bukan sekadar frekuensi)',
             fontsize=11, fontweight='bold')

mean_global = np.array(X_viz.mean(axis=0)).flatten()
top20       = mean_global.argsort()[-20:][::-1]
axes[0].barh(feats_viz[top20][::-1], mean_global[top20][::-1],
             color='#FF6F00', edgecolor='white')
axes[0].set_title('Top 20 Kata - Bobot TF-IDF Global', fontweight='bold')
axes[0].set_xlabel('Rata-rata Bobot TF-IDF')

label_cfg = [('positif','#4CAF50'), ('netral','#2196F3'), ('negatif','#F44336')]
offset = np.arange(5)
width  = 0.25
for i, (lbl, color) in enumerate(label_cfg):
    idx_lbl = df[df['label']==lbl].index
    mean_l  = np.array(X_viz[idx_lbl].mean(axis=0)).flatten()
    top5    = mean_l.argsort()[-5:][::-1]
    axes[1].bar(offset + i*width, mean_l[top5], width=width,
                color=color, alpha=0.85, label=lbl.capitalize(),
                edgecolor='white')
    for j, (pos, val) in enumerate(zip(offset + i*width, mean_l[top5])):
        axes[1].text(pos, val+0.0003, feats_viz[top5[j]],
                     ha='center', va='bottom', fontsize=7, rotation=45)

axes[1].set_title('Top 5 Bobot TF-IDF per Kelas', fontweight='bold')
axes[1].set_xlabel('Posisi Fitur')
axes[1].set_ylabel('Bobot TF-IDF Rata-rata')
axes[1].set_xticks(offset + width); axes[1].set_xticklabels([f'#{i+1}' for i in range(5)])
axes[1].legend()
plt.tight_layout(); plt.show()
print("\nCatatan: TF-IDF = Term Frequency x Inverse Document Frequency.")
print("Kata dengan bobot tinggi = sering muncul di dokumen tertentu tapi jarang di dokumen lain -> diskriminatif.")
print("Berbeda dengan frekuensi mentah yang hanya menghitung kemunculan tanpa mempertimbangkan distribusi.")


## 4. Feature Engineering - TF-IDF Unigram / Bigram / Trigram

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df['label'])
class_names = le.classes_
print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

X_raw_train, X_raw_test, y_train, y_test = train_test_split(
    df['text'], y, test_size=0.2, random_state=42, stratify=y
)
test_indices = X_raw_test.index

tfidf_uni = TfidfVectorizer(max_features=1000, ngram_range=(1,1), min_df=2)
tfidf_bi  = TfidfVectorizer(max_features=1000, ngram_range=(2,2), min_df=2)
tfidf_tri = TfidfVectorizer(max_features=1000, ngram_range=(3,3), min_df=2)

X_train_uni = tfidf_uni.fit_transform(X_raw_train)
X_test_uni  = tfidf_uni.transform(X_raw_test)

X_train_bi  = tfidf_bi.fit_transform(X_raw_train)
X_test_bi   = tfidf_bi.transform(X_raw_test)

X_train_tri = tfidf_tri.fit_transform(X_raw_train)
X_test_tri  = tfidf_tri.transform(X_raw_test)

print(f"\nTrain: {X_train_uni.shape[0]} | Test: {X_test_uni.shape[0]}")
print(f"Vocab unigram (dari train): {X_train_uni.shape[1]}")
print(f"Vocab bigram  (dari train): {X_train_bi.shape[1]}")
print(f"Vocab trigram (dari train): {X_train_tri.shape[1]}")


## 5. Definisi Model

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=1.0),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=10,
                                                   class_weight='balanced', random_state=42, n_jobs=-1),
    'SVM':                 SVC(kernel='linear', C=0.5, class_weight='balanced',
                               probability=True, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                          subsample=0.7, colsample_bytree=0.7,
                                          eval_metric='mlogloss', random_state=42),
}
print(f"Total model: {len(models)}")


## 6. Pelatihan & Evaluasi (Unigram, Bigram, Trigram)

In [ ]:
from sklearn.pipeline import Pipeline

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate(model, X_tr, X_te, vectorizer):
    model.fit(X_tr, y_train)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)

    pipe      = Pipeline([('tfidf', clone(vectorizer)), ('clf', clone(model))])
    cv_scores = cross_val_score(pipe, df['text'], y, cv=skf,
                                scoring='f1_weighted', n_jobs=-1)
    return {
        'y_pred':     y_pred,
        'y_proba':    y_proba,
        'Accuracy':   accuracy_score(y_test, y_pred),
        'Precision':  precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall':     recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1-Score':   f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'ROC-AUC':    roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'),
        'CV F1 Mean': cv_scores.mean(),
        'CV F1 Std':  cv_scores.std(),
    }

results_uni, results_bi, results_tri = {}, {}, {}

print("=== UNIGRAM ===")
for name, model in models.items():
    results_uni[name] = evaluate(clone(model), X_train_uni, X_test_uni, tfidf_uni)
    r = results_uni[name]
    print(f" {name:22s} F1={r['F1-Score']:.4f} AUC={r['ROC-AUC']:.4f} CV={r['CV F1 Mean']:.4f}±{r['CV F1 Std']:.4f}")

print("\n=== BIGRAM ===")
for name, model in models.items():
    results_bi[name] = evaluate(clone(model), X_train_bi, X_test_bi, tfidf_bi)
    r = results_bi[name]
    print(f" {name:22s} F1={r['F1-Score']:.4f} CV={r['CV F1 Mean']:.4f}±{r['CV F1 Std']:.4f}")

print("\n=== TRIGRAM ===")
for name, model in models.items():
    results_tri[name] = evaluate(clone(model), X_train_tri, X_test_tri, tfidf_tri)
    r = results_tri[name]
    print(f" {name:22s} F1={r['F1-Score']:.4f} CV={r['CV F1 Mean']:.4f}±{r['CV F1 Std']:.4f}")


## 7. Tabel Perbandingan Performa

In [ ]:
def make_table(results, title):
    exclude = ('y_pred', 'y_proba')
    tbl = pd.DataFrame({
        k: {m: v for m, v in v.items() if m not in exclude}
        for k, v in results.items()
    }).T.sort_values('F1-Score', ascending=False)
    print(f"\n{'='*75}")
    print(f"  {title}")
    print(f"{'='*75}")
    display(tbl.style
            .format('{:.4f}')
            .background_gradient(cmap='YlOrRd', subset=['Accuracy','F1-Score','ROC-AUC'])
            .set_caption(title))
    return tbl

tbl_uni = make_table(results_uni, 'TF-IDF Unigram')
tbl_bi  = make_table(results_bi,  'TF-IDF Bigram')
tbl_tri = make_table(results_tri, 'TF-IDF Trigram')


## 8. Visualisasi Perbandingan Semua Model & N-gram

In [ ]:
metric_cols = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC','CV F1 Mean']
fig, axes = plt.subplots(3, 6, figsize=(26, 14))
fig.suptitle('Perbandingan Performa - Unigram / Bigram / Trigram', fontsize=13, fontweight='bold')

for row_idx, (results, ng_label) in enumerate([
    (results_uni,'Unigram'),(results_bi,'Bigram'),(results_tri,'Trigram')
]):
    tbl = pd.DataFrame({k:{m:v for m,v in v.items() if m not in ('y_pred','y_proba')}
                         for k,v in results.items()}).T
    for col_idx, metric in enumerate(metric_cols):
        ax   = axes[row_idx][col_idx]
        vals = tbl[metric].sort_values()
        bc   = [MODEL_PALETTE[list(tbl.index).index(n)] for n in vals.index]
        bs   = ax.barh(vals.index, vals.values, color=bc, edgecolor='white', height=0.6)
        for bar, val in zip(bs, vals.values):
            ax.text(val+0.005, bar.get_y()+bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=7, fontweight='bold')
        ax.set_xlim(0,1.12); ax.set_title(f'{ng_label}\n{metric}', fontweight='bold', fontsize=8)

plt.tight_layout(); plt.show()


## 9. Confusion Matrix - Semua Model (Unigram)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Confusion Matrices - TF-IDF Unigram', fontsize=13, fontweight='bold')

for ax, (name, res) in zip(axes.flat, results_uni.items()):
    cm   = confusion_matrix(y_test, res['y_pred'])
    cmap = 'Oranges' if name == 'XGBoost' else 'Blues'
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, linecolor='white', cbar=False)
    ax.set_title(name, fontweight='bold', color='#FF6F00' if name=='XGBoost' else 'black')
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')

plt.tight_layout(); plt.show()


## 9.5 Confusion Matrix - Bigram & Trigram

In [ ]:
for results, ng_label in [(results_bi, 'Bigram'), (results_tri, 'Trigram')]:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'Confusion Matrices - TF-IDF {ng_label}', fontsize=13, fontweight='bold')
    for ax, (name, res) in zip(axes.flat, results.items()):
        cm   = confusion_matrix(y_test, res['y_pred'])
        cmap = 'Oranges' if name == 'XGBoost' else 'Blues'
        sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                    xticklabels=class_names, yticklabels=class_names,
                    linewidths=0.5, linecolor='white', cbar=False)
        ax.set_title(name, fontweight='bold',
                     color='#FF6F00' if name=='XGBoost' else 'black')
        ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
    plt.tight_layout(); plt.show()


## 10. ROC Curves (Macro OvR)

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0,1,2])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ROC Curves - Macro Average (OvR)', fontsize=13, fontweight='bold')

for ax, (results, label) in zip(axes, [
    (results_uni,'Unigram'),(results_bi,'Bigram'),(results_tri,'Trigram')
]):
    for (name, res), color in zip(results.items(), MODEL_PALETTE):
        fpr_d, tpr_d = {}, {}
        for ci in range(3):
            fpr_d[ci], tpr_d[ci], _ = roc_curve(y_test_bin[:,ci], res['y_proba'][:,ci])
        all_fpr  = np.unique(np.concatenate([fpr_d[i] for i in range(3)]))
        mean_tpr = np.mean([np.interp(all_fpr, fpr_d[i], tpr_d[i]) for i in range(3)], axis=0)
        auc_val  = roc_auc_score(y_test, res['y_proba'], multi_class='ovr', average='macro')
        ax.plot(all_fpr, mean_tpr, lw=2.5 if name=='XGBoost' else 1.2,
                ls='-' if name=='XGBoost' else '--',
                color=color, label=f'{name} ({auc_val:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=0.8); ax.set_title(label, fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=7.5, loc='lower right')

plt.tight_layout(); plt.show()


## 11. Feature Importance - XGBoost

In [ ]:
xgb_trained = clone(models['XGBoost'])
xgb_trained.fit(X_train_uni, y_train)

feat_names  = np.array(tfidf_uni.get_feature_names_out())
importances = xgb_trained.feature_importances_
top_idx = importances.argsort()[-25:][::-1]

fig, ax = plt.subplots(figsize=(12, 7))
colors_fi = ['#FF6F00' if i==top_idx[0] else '#FFCC80' for i in top_idx]
ax.barh(feat_names[top_idx][::-1], importances[top_idx][::-1],
        color=colors_fi[::-1], edgecolor='white')
ax.set_title('XGBoost - Top 25 Feature Importance (Unigram)', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.show()


## 11.5 Analisis Error Model - Contoh Misklasifikasi XGBoost

In [ ]:
y_pred_xgb = results_uni['XGBoost']['y_pred']
wrong_mask  = (y_test != y_pred_xgb)

raw_test_series = df.loc[test_indices, 'text'].reset_index(drop=True)

error_df = pd.DataFrame({
    'Teks (TEXT_SIAP)'  : raw_test_series[wrong_mask].values,
    'Label Aktual'      : le.inverse_transform(y_test[wrong_mask]),
    'Prediksi Model'    : le.inverse_transform(y_pred_xgb[wrong_mask]),
    'Panjang Token'     : [len(t.split()) for t in raw_test_series[wrong_mask].values],
})

print("=== ANALISIS MISKLASIFIKASI - XGBoost (Unigram) ===")
print(f"Total salah klasifikasi : {wrong_mask.sum()} dari {len(y_test)} data test")
print(f"Error rate              : {wrong_mask.sum()/len(y_test)*100:.2f}%")

print("\nDistribusi jenis kesalahan (Aktual -> Prediksi):")
err_cross = pd.crosstab(
    le.inverse_transform(y_test[wrong_mask]),
    le.inverse_transform(y_pred_xgb[wrong_mask]),
    rownames=['Aktual'], colnames=['Prediksi']
)
display(err_cross)

print(f"\n15 contoh tweet yang salah diklasifikasi:")
display(error_df.head(15).style
        .hide(axis='index')
        .set_properties(**{'text-align': 'left', 'white-space': 'normal', 'max-width': '500px'})
        .set_caption('Contoh Misklasifikasi XGBoost Unigram'))

print("\nRata-rata panjang token per pasangan (Aktual -> Prediksi):")
summary_err = (error_df.groupby(['Label Aktual','Prediksi Model'])['Panjang Token']
               .agg(Jumlah='count', Rata2Token='mean')
               .round(1).reset_index())
display(summary_err)


## 12. Radar Chart - Perbandingan Menyeluruh

In [ ]:
radar_m = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']
N = len(radar_m)
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]

fig = plt.figure(figsize=(8,8))
ax  = fig.add_subplot(111, polar=True)

tbl_uni_r = pd.DataFrame({k:{m:v for m,v in v.items() if m not in ('y_pred','y_proba')}
                            for k,v in results_uni.items()}).T

for (mname, row), mc in zip(tbl_uni_r[radar_m].iterrows(), MODEL_PALETTE):
    vals_r = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, vals_r, 'o-', lw=2.5 if mname=='XGBoost' else 1.2, color=mc, label=mname)
    ax.fill(angles, vals_r, alpha=0.2 if mname=='XGBoost' else 0.04, color=mc)

ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_m, fontsize=11)
ax.set_ylim(0,1)
ax.set_title('Radar Chart - Perbandingan Model (Unigram)', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15))
plt.show()


## 13. Classification Report - XGBoost

In [ ]:
print("="*60)
print("XGBoost (Unigram) - Classification Report")
print("="*60)
print(classification_report(y_test, results_uni['XGBoost']['y_pred'],
                             target_names=class_names, digits=4))

rep    = classification_report(y_test, results_uni['XGBoost']['y_pred'],
                                target_names=class_names, output_dict=True)
rep_df = pd.DataFrame(rep).T.loc[class_names, ['precision','recall','f1-score']].astype(float)

fig, ax = plt.subplots(figsize=(7,3))
sns.heatmap(rep_df, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white', vmin=0.5, vmax=1.0)
ax.set_title('XGBoost - Classification Report Heatmap', fontweight='bold')
plt.tight_layout(); plt.show()


## 13.5 Pengujian Statistik - Cross-Validation Detail per Model

In [ ]:
print("=== HASIL CROSS-VALIDATION 5-FOLD - UNIGRAM ===")
print(f"{'Model':<22} {'CV F1 Mean':>12} {'CV F1 Std':>12} {'Test F1':>10} {'Gap':>8}")
print("-" * 70)
for name, res in results_uni.items():
    gap = res['F1-Score'] - res['CV F1 Mean']
    print(f"{name:<22} {res['CV F1 Mean']:>12.4f} {res['CV F1 Std']:>12.4f}"
          f" {res['F1-Score']:>10.4f} {gap:>+8.4f}")

print("\nKeterangan:")
print("  CV F1 Mean = rata-rata F1 pada 5-fold cross-validation (lebih representatif)")
print("  CV F1 Std  = standar deviasi antar fold (lebih kecil = lebih stabil)")
print(" Gap (+) = test F1 > CV F1 -> mungkin kebetulan (split yang menguntungkan)")
print(" Gap (-) = test F1 < CV F1 -> test set lebih sulit dari rata-rata")

model_names = list(results_uni.keys())
means   = [results_uni[n]['CV F1 Mean'] for n in model_names]
stds    = [results_uni[n]['CV F1 Std']  for n in model_names]
test_f1 = [results_uni[n]['F1-Score']   for n in model_names]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(model_names))
ax.bar(x, means, yerr=stds, color=MODEL_PALETTE, edgecolor='white',
       capsize=6, alpha=0.85, label='CV F1 Mean ± Std')
ax.scatter(x, test_f1, color='black', zorder=5, s=70, marker='D', label='Test F1-Score')

for i, (mean, std, tf) in enumerate(zip(means, stds, test_f1)):
    ax.text(i, mean + std + 0.012, f'{mean:.3f}\n±{std:.3f}',
            ha='center', fontsize=7.5, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(0, 1.08)
ax.set_ylabel('F1-Score (Weighted)')
ax.set_title('Cross-Validation F1 Mean ± Std vs Test F1-Score (Unigram)', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()


## 14. Kesimpulan

### Mengapa Hasilnya Tidak 100%?
- Dataset menggunakan **label gabungan 3 anotator (majority vote)** -> ada ketidaksepakatan antar anotator yang membuat beberapa sampel ambigu
- Teks Twitter/media sosial mengandung **sarkasme, bahasa campuran, dan konteks implisit** yang sulit dideteksi hanya dengan TF-IDF
- Regularisasi model mencegah overfitting -> skor CV ~= skor test

### Perbandingan Antar N-gram
| N-gram | Karakteristik |
|--------|---------------|
| **Unigram** | Paling stabil, vocab terbesar, terbaik untuk teks pendek |
| **Bigram**  | Menangkap frasa kunci (*judi online*, *slot gacor*), sedikit lebih sparse |
| **Trigram** | Paling sparse, performa turun signifikan pada data kecil |

### Model Terbaik
**Logistic Regression** dan **SVM** sedikit unggul di Unigram (karena regularisasi linear cocok untuk TF-IDF sparse).  
**XGBoost** menunjukkan keunggulan pada **CV consistency** dan tetap kompetitif - lebih unggul saat data bertambah besar atau fitur lebih kompleks.

### Cara Deploy
```python
import joblib
pipeline = {'tfidf': tfidf_uni, 'model': xgb_trained, 'label_encoder': le}
joblib.dump(pipeline, 'judol_pipeline.pkl')
```
